# SETUP

In [ ]:
### IMPORTS

from pathlib import Path
import logging
import rdflib
import sys
from sentence_transformers import SentenceTransformer
import chromadb

# CONFIG

In [ ]:

ONTOLOGY_PATH = Path("ontology.rdf")
ONTOLOGY_FORMAT = "xml"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

#CHROMA_COLLECTION_NAME = "ODD_embeddings"

logging.basicConfig(
    filename='ODD-Log.log',
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s – %(message)s",
)
logger = logging.getLogger("ODD-RAG")

# CHUNKING

Teilt die Ontologie in kleinere Subgraphen auf. Dabei wird für jede Entität ein eigenes Dict erstellt, welches mit den zur Entität gehörenden Triples gefüllt wird.

To-Do: Checken, ob Annotations auch Sinn ergeben. Ich glaube aber in der SUNRISE wurden keine verwendet.

In [ ]:
OWL_ENTITY_TYPES = {
    rdflib.OWL.Class,
    rdflib.OWL.ObjectProperty,
    rdflib.OWL.DatatypeProperty,
    rdflib.OWL.NamedIndividual,
}

def get_label(graph: rdflib.Graph, uri: rdflib.URIRef) -> str:
    """
    Gibt das Label einer URI zurück.
    Wenn kein Label vergeben ist, wird stattdessen der Name aus der URI genommen.
    """

    for _, _, obj in graph.triples((uri, rdflib.namespace.RDFS.label, None)):
        return str(obj) 
    uri_name = str(uri).rsplit("#", 1)[-1].rsplit("/", 1)[-1]
    return uri_name

def chunk_ontology(graph: rdflib.Graph) -> list[dict]:
    """
    Zerlegt die gegebene Ontologie in Chunks, wobei jeder Eintrag einer Entität entspricht.
    Chunks enthalten:
    - URI
    - Label
    - Entitätstyp, z.B. Klasse, ObjectProperty, etc.
    - Turtle-Serialisierung der Triple dieser Entität
    - Beschreibungstext für Embedding
    """

    chunks = []
    entities = {}   # ein Dict für jede Entität

    # Dict aller Entitäten mit ihren Typen
    # Keys sind die URIs, Values der Typ
    for entity_type in OWL_ENTITY_TYPES:
        for subj, pred, obj in graph.triples((None, rdflib.RDF.type, entity_type)):
            if isinstance(subj, rdflib.URIRef):
                entities[subj] = entity_type.toPython().rsplit("#", 1)[-1]

    # Erstellt Subgraph für jede Entität
    for uri, e_type in entities.items():
        sub_graph = rdflib.Graph()
        
        for triple in graph.triples((uri, None, None)):
            sub_graph.add(triple)

        # Serialisiert den Subgraphen einer jeden Entität
        #turtle_str = sub_graph.serialize(format="turtle")
        #turtle_str = ""
        label = get_label(graph, uri)

        text_parts = [f"Entität: {label} (Typ: {e_type})"]
        for _, pred, obj in sub_graph.triples((uri, None, None)):
            pred_local = str(pred).rsplit("#", 1)[-1].rsplit("/", 1)[-1]
            obj_str = str(obj).rsplit("#", 1)[-1].rsplit("/", 1)[-1]
            text_parts.append(f"{pred_local}: {obj_str}")

        chunks.append({
            "chunk_id":    str(uri),
            "label":       label,
            "entity_type": e_type,
            #"turtle":      turtle_str,
            "text":        "\n".join(text_parts),
        })

    logger.info("Chunking abgeschlossen: %d Entitäts-Chunks erzeugt.", len(chunks))
    return chunks


if not ONTOLOGY_PATH.exists():
    logger.warning(f"Ontologie-Datei '{ONTOLOGY_PATH}' nicht gefunden.")
    sys.exit(f"Ontologie-Datei '{ONTOLOGY_PATH}' nicht gefunden.")

logger.info(f"Lade Ontologie '{ONTOLOGY_PATH}'")
onto = rdflib.Graph()
onto.parse(str(ONTOLOGY_PATH))
chunks = chunk_ontology(onto)

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("Modell geladen")

In [ ]:

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(
    name="test",
    embedding_function=None
)
print("ok")

In [ ]:
collection.add(
    ids=["1"],
    embeddings=[[0.1, 0.2, 0.3]]
)

# EMBEDDING

Die einzelnen Chunks werden eingebettet, sodass sie vom LLM verarbeitet werden können.

To-Do: Vielleicht List-Comprehension anstatt For-Loop?

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
client = chromadb.PersistentClient(path="./chroma_db")

COLLECTION_NAME = "odd_embeddings"

try:
    client.delete_collection(name=COLLECTION_NAME) 
    collection = client.create_collection(name=COLLECTION_NAME)
except:
    collection = client.create_collection(name=COLLECTION_NAME)


collection.add(
    embeddings=[[1,2,3],[1,2,3],[1,2,3]],
    ids=["1","2","3"]
)

texts = []
ids = []
metas = []
embeddings = []

for index, entity in enumerate(chunks):

    text = entity["text"]
    texts.append(text)
    id = f"chunk_{index}"
    ids.append(id)
    """
    meta_data = {
        "chunk_id": entity["chunk_id"],
        "label": entity["label"],
        "entity_type": entity["entity_type"],
        #"turtle": entity["turtle"],
    }
    #metas.append(meta_data)
    """
    
print(len(texts))
batch_size = 32

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]
    batch_metas = metas[i:i+batch_size]

    print(f"ich arbeite an Batch {i}")

    batch_embeddings = embedder.encode(batch_texts, device="cpu")
    collection.add(
    ids=batch_ids,
    embeddings=batch_embeddings.tolist(),
    documents=batch_texts,
    metadatas=batch_metas
    )
    


    print(f"Batch {i} verarbeitet")
#embedding = embedder.encode(texts, show_progress_bar=True, batch_size=4)
#embeddings.append(embedding)

print("encoding klappt auch")

#collection.add(
#ids = ids,
#embeddings=embedding,
#documents=texts,
#metadatas=metas
#)

logger.info(f"Vektordatenbank befüllt: {len(chunks)} Chunks wurden geladen.")


logger.info(f"{len(chunks)} Chunks wurden eingebettet.")

# VECTOR DATABASE

Die eingebetteten Textpassagen werden in einer Vektordatenbank abgespeichert.